In [ ]:
import json
import math
import re
from typing import cast

import folium
import geopandas as gpd
import pandas as pd
import requests
import shapely
from branca.element import Figure
from folium.plugins import Draw, MeasureControl, MousePosition
from shapely.geometry import LineString, Point, shape
from shapely.ops import linemerge, nearest_points, substring

from tunnel.config import (
    BNG,
    CATCHMENT_M,
    COLOURS,
    DERIVED,
    NORTH_PORTAL,
    ROOT,
    SOUTH_PORTAL,
    WGS84,
    catchment,
    to_bng,
    tunnel_line,
)
from tunnel.data import stations, thames
from tunnel.ownership import GROUPS, rotherhithe_side

# Crossing map

> An interactive map of the crossing to illustrate the blog: who owns the land, the proposed tunnel, and the tunnels already under this stretch of river.

One interactive map of the crossing, with every layer saved as GeoJSON in `../crossing_map/layers/`
so it can be toggled on the map, opened in QGIS, or reused in other notebooks.

- **Ownership:** HMLR plots in the readjustment area (the Rotherhithe side within `CATCHMENT_M` of
  Durand's Wharf), saved by the ownership notebook and grouped as there.
- **Proposed tunnel:** Durand's Wharf to the West India Docks impound lock, as in the blog.
- **Tunnels:** every tunnel layer is clipped to one circle, centred on the middle of the TfL 2017
  immersed tube and just big enough to take in the Woolwich foot tunnel (`TUNNEL_AREA`, about 6.5 km).
  Inside it: road, rail and foot tunnels, the abandoned Thames Archway driftway, and the Tideway and
  Lee sewer tunnels.
- **TfL 2017 tunnel option:** the immersed tube TfL shortlisted, on the dock-to-dock route IanVisits described.

Every feature carries `source` and `confidence`. **Indicative** means drawn by hand from written
descriptions rather than taken from a published dataset or drawing; it shows in each feature's popup.

**On the map:** solid lines are built, dashed proposed, dotted abandoned. Line width scales with
the bore width (`width_m`, sourced in `WIDTHS`).

**Adding layers:** save any GeoJSON (WGS84) into `crossing_map/layers/` and re-run the last section; it
appears with a default style. **Annotating:** draw on the map, click _Export_, move the downloaded
`annotations.geojson` into `layers/` and re-run.


In [ ]:
MAP_DIR = ROOT / "crossing_map"
LAYERS = MAP_DIR / "layers"
SOURCES = MAP_DIR / "sources"
for _d in (LAYERS, SOURCES):
    _d.mkdir(parents=True, exist_ok=True)


# Line style on the map: dashed = proposed, dotted = abandoned, solid = built.
# Line width scales with the bore width below.
LAYER_STATUS = {
    "tunnels_existing": "built",
    "tunnels_historic": "abandoned",
    "tunnels_utility": "built",
}  # everything else: proposed

# (name prefix, width in metres, where the figure comes from). Twin-bore tunnels give one bore.
WIDTHS = [
    (
        "Jubilee line",
        4.35,
        "Wikipedia: Jubilee line extension, tunnel diameter (twin bores)",
    ),
    (
        "Thames Tunnel",
        11.0,
        "Wikipedia: Thames Tunnel, 35 ft wide overall (two arches)",
    ),
    ("DLR", 5.2, "assumed: typical DLR bored tunnel (twin bores)"),
    (
        "London Overground: East London",
        5.0,
        "assumed: typical cut-and-cover double track (twin bores)",
    ),
    ("Elizabeth line", 6.2, "Wikipedia: Crossrail, internal diameter (twin bores)"),
    ("Rotherhithe Tunnel", 9.35, "Wikipedia: Rotherhithe Tunnel, shield diameter"),
    (
        "Greenwich foot tunnel",
        2.74,
        "Wikipedia: Greenwich foot tunnel, internal diameter",
    ),
    ("Woolwich foot tunnel", 2.74, "assumed: built to the Greenwich model"),
    (
        "Blackwall Tunnel western",
        8.23,
        "Wikipedia: Blackwall Tunnel, external diameter",
    ),
    ("Blackwall Tunnel eastern", 8.59, "Wikipedia: Blackwall Tunnel, diameter"),
    ("Silvertown", 11.87, "Wikipedia: Silvertown Tunnel, TBM diameter (twin bores)"),
    ("Thames Archway", 1.5, "assumed: small pilot driftway"),
    (
        "Thames Tideway Tunnel: Greenwich",
        5.0,
        "Tideway DCO 2014, Work No.20: internal diameter",
    ),
    (
        "Thames Tideway Tunnel",
        7.2,
        "Tideway DCO 2014, Work No.1: internal diameter (6.5 m west of Carnwath Road)",
    ),
    ("Lee Tunnel", 7.2, "Thames Water: Lee Tunnel internal diameter"),
    ("Proposed tunnel", 7.0, "assumed: about 7 m outside diameter, as in the cost estimate"),
    (
        "TfL 2017 immersed tube",
        8.0,
        "assumed: as the TfL 2019 bridge deck; TfL gave no size",
    ),
]


def ll(lon, lat):
    return gpd.GeoSeries([Point(lon, lat)], crs=WGS84).to_crs(BNG)[0].coords[0]


def save(features: list[dict], name: str, crs: str = BNG) -> None:
    if name in TUNNEL_LAYERS:
        features = [{**f, "geom": f["geom"].intersection(TUNNEL_AREA)} for f in features]
        features = [f for f in features if not f["geom"].is_empty]
    for f in features:
        f["props"].setdefault("status", LAYER_STATUS.get(name, "proposed"))
        for prefix, width, src in WIDTHS:
            if f["props"].get("name", "").startswith(prefix) and f["geom"].geom_type != "Point":
                f["props"].setdefault("width_m", width)
                f["props"].setdefault("width_source", src)
                break
    gdf = gpd.GeoDataFrame([f["props"] for f in features], geometry=[f["geom"] for f in features], crs=crs)
    gdf.to_crs(WGS84).to_file(LAYERS / f"{name}.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)
    print(f"  {name}.geojson: {len(gdf)} features")


def feat(geom, **props) -> dict:
    return {"geom": geom, "props": props}

## Tunnel circle

Every tunnel layer is clipped to `TUNNEL_AREA`: a circle centred halfway along the TfL 2017 immersed
tube, with a radius just long enough to take in the whole Woolwich foot tunnel. Tunnels that run on
beyond it (the Jubilee and Elizabeth lines, the Tideway) are drawn up to its edge.


In [ ]:
# TfL 2017 immersed tube ramps (BNG): Nelson Docks basin mouth and head, from its OpenStreetMap
# outline (way 52706575), and the impound lock inlet centre line, from the river outline (way 199827370)
NELSON_MOUTH, NELSON_HEAD = (536660, 180183), (536565, 180175)
INLET_MOUTH, INLET_HEAD = (537036, 180103), (537111, 180094)
WOOLWICH_FOOT_TUNNEL = LineString([ll(0.06165, 51.49879), ll(0.06289, 51.49432)])

TUNNEL_CENTRE = LineString([NELSON_HEAD, NELSON_MOUTH, INLET_MOUTH, INLET_HEAD]).interpolate(0.5, normalized=True)
TUNNEL_RADIUS_M = TUNNEL_CENTRE.hausdorff_distance(WOOLWICH_FOOT_TUNNEL) + 10
TUNNEL_AREA = TUNNEL_CENTRE.buffer(TUNNEL_RADIUS_M, quad_segs=64)
TUNNEL_LAYERS = {
    "tunnels_existing",
    "tunnels_historic",
    "tunnels_utility",
    "tunnel_tfl_2017",
    "tunnel_proposed",
}
print(f"Tunnel circle: {TUNNEL_RADIUS_M / 1000:.2f} km radius")

Tunnel circle: 6.53 km radius


## Ownership layer

Title plans in the readjustment area, from `data/derived/catchment_plots.gpkg` (written by the ownership
notebook), clipped to the Rotherhithe side of the river centre line. Plots mostly covered by a larger
one (sub-titles, leases) are dropped.


In [ ]:
KEEP = ["title_no", "tenure", "owner", "owner_category", "address", "company_no", "area_m2"]

g = gpd.read_file(DERIVED / "catchment_plots.gpkg")
g = g[g.top_level].copy()
g = gpd.clip(g, rotherhithe_side(catchment()))
# Clipping can leave GeometryCollections (polygon plus slivers), which
# Leaflet can't attach tooltips to: keep only the polygonal parts.
g["geometry"] = g.geometry.apply(
    lambda geom: shapely.union_all(
        [p for p in getattr(geom, "geoms", [geom]) if p.geom_type in ("Polygon", "MultiPolygon")]
    )
)
g = g[~g.geometry.is_empty]

# Big plots first so smaller ones sit on top; freeholds over leases on the same footprint
g["freehold"] = g.tenure.eq("Freehold")
g = g.sort_values(["area_m2", "freehold"], ascending=[False, True])

g["owner_group"] = g.group.map(GROUPS)
g["colour"] = g.group.map(COLOURS)
g["area_m2"] = g.area_m2.round()
g = g[KEEP + ["group", "owner_group", "colour", "geometry"]]
g["geometry"] = g.geometry.simplify(0.2)
g.to_crs(4326).to_file(LAYERS / "ownership.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)
print(f"  ownership.geojson: {len(g)} plots")

  ownership.geojson: 1709 plots


## Existing and historic tunnels

Rail tunnels are the Jubilee, Elizabeth, DLR and East London lines wherever they run underground
inside the circle: the line comes from Oliver O'Brien's `tfl_lines.json`, kept where OpenStreetMap tags
the track as tunnel (gaps of up to 400 m, mostly underground stations, are filled). Both are cached in
`sources/`.
The Rotherhithe, Blackwall and Silvertown road tunnels come straight from OpenStreetMap (also cached
in `sources/`); the foot tunnels use their published entrance coordinates. The Thames Archway driftway is inferred from
descriptions.


In [ ]:
# OSM-derived line geometry for the TfL network, by Oliver O'Brien (tubecreature).
TFL_LINES_URL = "https://raw.githubusercontent.com/oobrien/vis/master/tubecreature/data/tfl_lines.json"
TFL_LINES = SOURCES / "tfl_lines.json"
TFL_LINES_SRC = "OpenStreetMap via github.com/oobrien/vis (tfl_lines.json)"

# tfl_lines.json line name -> (OpenStreetMap tunnel way name pattern, feature name, opened, note)
RAIL_LINES = {
    "Jubilee": (
        r"Jubilee",
        "Jubilee line",
        1999,
        "Jubilee line extension: twin bored tunnels from Westminster to Canning Town, under the river four times",
    ),
    "Elizabeth line": (
        r"Elizabeth",
        "Elizabeth line",
        2022,
        "Crossrail twin bored tunnels, completed 2022. Canary Wharf station is in the North Dock",
    ),
    "DLR": (
        r"Docklands Light Railway",
        "DLR",
        1991,
        "Bank branch (1991), Lewisham extension under the river (1999) and short covered sections near Stratford",
    ),
    "London Overground": (
        r"East London|Thames Tunnel",
        "London Overground: East London line",
        1869,
        "Whitechapel to Surrey Quays, mostly cut and cover, through Brunel's Thames Tunnel",
    ),
}
_lon, _lat = gpd.GeoSeries([TUNNEL_CENTRE], crs=BNG).to_crs(WGS84)[0].coords[0]
RAIL_TUNNELS_URL = "https://maps.mail.ru/osm/tools/overpass/api/interpreter?data=" + requests.utils.quote(
    '[out:json][timeout:120];way["railway"~"^(subway|rail|light_rail)$"]["tunnel"="yes"]'
    f"(around:{TUNNEL_RADIUS_M + 30:.0f},{_lat:.5f},{_lon:.5f});out tags geom;"
)


def rail_tunnel_ways(pattern: str):
    """OpenStreetMap railway tunnel ways in the circle whose name matches `pattern`, in BNG."""
    cache = SOURCES / "rail_tunnels_osm.json"
    if not cache.exists():
        r = requests.get(
            RAIL_TUNNELS_URL,
            headers={"User-Agent": "tunnel-stuff crossing map"},
            timeout=180,
        )
        r.raise_for_status()
        cache.write_text(r.text)
    ways = [
        LineString([ll(pt["lon"], pt["lat"]) for pt in e["geometry"]])
        for e in json.loads(cache.read_text())["elements"]
        if re.search(pattern, e["tags"].get("name", ""))
    ]
    return shapely.union_all(ways)


def tunnel_sections(line_name: str, pattern: str, max_gap: float = 400, min_len: float = 200):
    """The line's route where OpenStreetMap tags the track as tunnel, with gaps of up to
    `max_gap` metres (underground stations, crossovers) filled in. Gaps under the Thames are
    always filled: TfL's route is schematic there and can stray 100 m+ from the bores."""
    route = [
        shape(f["geometry"])
        for f in tfl_lines()["features"]
        if any(line["name"] == line_name for line in f["properties"]["lines"])
    ]
    route = gpd.GeoSeries(route, crs=WGS84).to_crs(BNG).union_all()
    route = shapely.line_merge(route.intersection(TUNNEL_AREA.buffer(2000)))
    near_tunnel = rail_tunnel_ways(pattern).buffer(40)
    river = thames().union_all()
    sections = []
    for r in shapely.get_parts(route):
        spans = sorted(
            sorted(r.project(Point(c)) for c in (part.coords[0], part.coords[-1]))
            for part in shapely.get_parts(r.intersection(near_tunnel))
            if part.geom_type == "LineString" and not part.is_empty
        )
        merged = []
        for a, b in spans:
            if merged and (a - merged[-1][1] <= max_gap or substring(r, merged[-1][1], a).intersects(river)):
                merged[-1][1] = max(merged[-1][1], b)
            else:
                merged.append([a, b])
        sections += [substring(r, a, b) for a, b in merged if b - a >= min_len]
    return shapely.union_all(sections)


OVERPASS_URL = "https://overpass-api.de/api/interpreter"


def osm_tunnel_bores(name: str) -> list[dict]:
    """Ways named `name` and tagged tunnel=yes in east London, cached in sources/."""
    cache = SOURCES / f"{name.lower().replace(' ', '_')}_osm.json"
    if not cache.exists():
        query = f'[out:json][timeout:30];way["tunnel"="yes"]["name"="{name}"](51.49,-0.07,51.52,0.03);out geom tags;'
        r = requests.post(
            OVERPASS_URL,
            data={"data": query},
            headers={"User-Agent": "tunnel-stuff crossing map"},
            timeout=60,
        )
        r.raise_for_status()
        cache.write_text(r.text)
    return json.loads(cache.read_text())["elements"]


def tfl_lines() -> dict:
    if not TFL_LINES.exists():
        SOURCES.mkdir(exist_ok=True)
        TFL_LINES.write_bytes(requests.get(TFL_LINES_URL, timeout=60).content)
    return json.loads(TFL_LINES.read_text())

In [ ]:
feats = []
for line_name, (pattern, name, opened, note) in RAIL_LINES.items():
    geom = tunnel_sections(line_name, pattern)
    if line_name == "London Overground":
        # Brunel's Thames Tunnel gets its own feature (and width)
        brunel = rail_tunnel_ways(r"^Thames Tunnel$").buffer(40)
        feats.append(
            feat(
                geom.intersection(brunel),
                name="Thames Tunnel (London Overground): Wapping to Rotherhithe",
                line=line_name,
                opened=1843,
                mode="rail",
                note="Brunel's tunnelling shield tunnel, the first under a navigable river. A foot tunnel until 1865",
                source=f"{TFL_LINES_SRC}; tunnel extent from OpenStreetMap",
                confidence="published",
            )
        )
        geom = geom.difference(brunel)
    feats.append(
        feat(
            geom,
            name=name,
            line=line_name,
            opened=opened,
            mode="rail",
            note=note,
            source=f"{TFL_LINES_SRC}; tunnel extent from OpenStreetMap",
            confidence="published",
        )
    )

(rotherhithe,) = osm_tunnel_bores("Rotherhithe Tunnel")
feats.append(
    feat(
        LineString([ll(pt["lon"], pt["lat"]) for pt in rotherhithe["geometry"]]),
        name="Rotherhithe Tunnel (road)",
        opened=1908,
        mode="road",
        note="1,482 m long, carriageway 15 m below high water. Shared with cyclists and pedestrians",
        source=f"OpenStreetMap way {rotherhithe['id']}",
        confidence="published",
    )
)
feats.append(
    feat(
        LineString([ll(-0.0093, 51.4866), ll(-0.0102, 51.4833)]),
        name="Greenwich foot tunnel",
        opened=1902,
        mode="foot",
        note="370 m long, 15 m deep, lifts and stairs in each shaft. The closest precedent for a foot tunnel here",
        source="Wikipedia: Greenwich foot tunnel (entrance coordinates)",
        confidence="published",
    )
)
feats.append(
    feat(
        WOOLWICH_FOOT_TUNNEL,
        name="Woolwich foot tunnel",
        opened=1912,
        mode="foot",
        note="504 m long, built to the Greenwich model. Run jointly by Greenwich, Tower Hamlets and Newham",
        source="Wikipedia: Woolwich foot tunnel (entrance coordinates)",
        confidence="published",
    )
)
# Blackwall Tunnel: the 1897 western bore is the longer, bendier one, lined up with
# Northumberland and Ordnance Wharves; the 1967 eastern bore runs straighter.
BLACKWALL_BORES = {
    "northbound": (
        "Blackwall Tunnel western bore (northbound)",
        1897,
        "1,350 m, sharp bends to line up with Northumberland and Ordnance Wharves. Originally also carried pedestrians",
    ),
    "southbound": (
        "Blackwall Tunnel eastern bore (southbound)",
        1967,
        "1,174 m, no sharp bends",
    ),
}
for way in osm_tunnel_bores("Blackwall Tunnel"):
    coords = [ll(pt["lon"], pt["lat"]) for pt in way["geometry"]]
    heading = "southbound" if coords[0][1] > coords[-1][1] else "northbound"
    name, opened, note = BLACKWALL_BORES[heading]
    feats.append(
        feat(
            LineString(coords),
            name=name,
            opened=opened,
            mode="road",
            note=note,
            source=f"OpenStreetMap way {way['id']}",
            confidence="published",
        )
    )
for way in osm_tunnel_bores("Silvertown Tunnel"):
    coords = [ll(pt["lon"], pt["lat"]) for pt in way["geometry"]]
    heading = "southbound" if coords[0][1] > coords[-1][1] else "northbound"
    feats.append(
        feat(
            LineString(coords),
            name=f"Silvertown Tunnel {heading} bore (road)",
            opened=2025,
            mode="road",
            note="1.4 km twin bores, 11.87 m TBM, opened April 2025. About GBP 2bn",
            source=f"OpenStreetMap way {way['id']}",
            confidence="published",
        )
    )
save(feats, "tunnels_existing")

# Abandoned Thames Archway Company driftway (1805-09): shaft near the end of
# Lavender Street, aiming for the Regent's Canal Dock entrance at Limehouse.
# It got within about 117 ft of the north bank before flooding ended it.
shaft, target = ll(-0.0378, 51.5068), ll(-0.0365, 51.5093)
driftway = LineString([shaft, target])
driftway = LineString([shaft, driftway.interpolate(driftway.length - 36)])  # stop ~117 ft short
save(
    [
        feat(
            Point(shaft),
            name="Thames Archway shaft (c.70 ft deep)",
            opened=1805,
            mode="abandoned",
            note="Sunk 'at the apex of Rotherhithe, near the end of Lavender Street'; now under Sovereign View",
            source="russiadock.blogspot.com: Before the Thames Tunnel",
            confidence="indicative",
        ),
        feat(
            driftway,
            name="Thames Archway driftway (abandoned 1808)",
            opened=1805,
            mode="abandoned",
            note="Vazie, then Trevithick. Pilot tunnel got within about 117 ft of the Limehouse bank before floods",
            source="russiadock.blogspot.com: Before the Thames Tunnel",
            confidence="indicative",
        ),
        feat(
            Point(target),
            name="Thames Archway intended north end, Regent's Canal Dock",
            mode="abandoned",
            note="Never reached",
            source="russiadock.blogspot.com: Before the Thames Tunnel",
            confidence="indicative",
        ),
    ],
    "tunnels_historic",
)

  tunnels_existing.geojson: 12 features
  tunnels_historic.geojson: 3 features


## Sewer tunnels

**Thames Tideway Tunnel** (the "super sewer"): Tideway hasn't published its alignment, but the City of
London's INSPIRE map service carries the planning safeguarding corridor (about 28 m wide) for every
tunnel in the scheme. The line drawn here is that corridor's centre line, so it is within about 14 m of
the safeguarded route, not the as-built tunnel. Where shaft sites break the corridor, the ends are joined
straight across. The data has no corridor for the 1.4 km under the river between Westminster and
Blackfriars, so that stretch follows the river's centre line from OpenStreetMap.

**Lee Tunnel** (2016): the 6.9 km tunnel that takes the Tideway's flows from Abbey Mills to Beckton. Its
alignment isn't published either, so it's a straight line between its end shafts.


In [ ]:
TIDEWAY_URL = (
    "https://www.mapping.cityoflondon.gov.uk/arcgis/rest/services/INSPIRE/MapServer/129/query"
    "?where=1%3D1&outFields=*&outSR=27700&f=geojson"
)
TIDEWAY_SRC = "Thames Tideway Tunnel safeguarding area (City of London INSPIRE map service); centre line derived"
# River Thames centre line from Westminster to Blackfriars, where the safeguarding data has no corridor
THAMES_CENTRAL_URL = "https://maps.mail.ru/osm/tools/overpass/api/interpreter?data=" + requests.utils.quote(
    '[out:json][timeout:50];way["waterway"="river"]["name"="River Thames"](51.500,-0.128,51.513,-0.098);out geom;'
)


def cached(path, url: str) -> str:
    if not path.exists():
        r = requests.get(url, headers={"User-Agent": "tunnel-stuff crossing map"}, timeout=120)
        r.raise_for_status()
        path.write_text(r.text)
    return path.read_text()


def centreline(corridor, spacing: float = 5.0):
    """Medial axis of a long thin polygon: the Voronoi edges of its outline that lie wholly inside it."""
    outline = shapely.MultiPoint(shapely.get_coordinates(shapely.segmentize(corridor.boundary, spacing)))
    inside = corridor.buffer(-0.5)
    edges = [e for e in shapely.get_parts(shapely.voronoi_polygons(outline, only_edges=True)) if inside.contains(e)]
    return shapely.line_merge(shapely.union_all(edges)).simplify(2)


def join_ends(parts: list, river, straight: float = 400, along_river: float = 2000) -> list:
    """Links from each loose end to the nearest loose end of another part: straight across shaft
    sites, or along the river centre line for longer gaps in the safeguarding data."""
    ends = [(i, Point(p.coords[k])) for i, p in enumerate(parts) for k in (0, -1)]
    loose = [(i, pt) for i, pt in ends if not any(j != i and pt.distance(q) < 0.5 for j, q in ends)]
    links = []
    for i, pt in loose:
        gap, q = min(((pt.distance(q), q) for j, q in loose if j != i), key=lambda t: t[0])
        if gap <= straight:
            links.append((i, LineString([pt, q])))
        elif gap <= along_river:
            reach = substring(river, river.project(pt), river.project(q))
            links.append((i, LineString([pt, *reach.coords, q])))
    return links


def tideway_branch(line) -> str:
    x, y = line.interpolate(0.5, normalized=True).coords[0]
    if x > 534250 and y < 179700:
        return "greenwich"
    return "main"


TIDEWAY_BRANCHES = {
    "main": (
        "Thames Tideway Tunnel (main tunnel)",
        (
            "25 km storage and transfer sewer from Acton to Abbey Mills, where it joins the Lee Tunnel to Beckton. "
            "Built 2016-25; catches overflows from 34 combined sewer outfalls"
        ),
    ),
    "greenwich": (
        "Thames Tideway Tunnel: Greenwich connection tunnel",
        "4.6 km from Greenwich Pumping Station to the main tunnel at Chambers Wharf, under Deptford",
    ),
}

safeguarding = gpd.read_file(cached(SOURCES / "tideway_safeguarding.geojson", TIDEWAY_URL))
corridors = safeguarding[safeguarding.TYPE == "Safeguarding for tunnels"].union_all()
# Beckton belongs to the older Lee Tunnel; slivers under 1000 m2 are drawing artefacts
corridors = [c for c in corridors.geoms if c.area > 1000 and c.bounds[0] < 540000]
parts = [p for c in corridors for p in shapely.get_parts(centreline(c)) if p.length > 20]

river_central = shapely.line_merge(
    shapely.union_all(
        [
            LineString([ll(pt["lon"], pt["lat"]) for pt in way["geometry"]])
            for way in json.loads(cached(SOURCES / "thames_central_osm.json", THAMES_CENTRAL_URL))["elements"]
        ]
    )
)

pieces = {k: [] for k in TIDEWAY_BRANCHES}
for p in parts:
    pieces[tideway_branch(p)].append(p)
for i, link in join_ends(parts, river_central):
    pieces[tideway_branch(parts[i])].append(link)

feats = [
    feat(
        shapely.line_merge(shapely.union_all(pieces[k])),
        name=name,
        opened=2025,
        mode="sewer",
        note=note,
        source=TIDEWAY_SRC,
        confidence="published corridor; centre line within about 14 m",
    )
    for k, (name, note) in TIDEWAY_BRANCHES.items()
]
# Lee Tunnel: no published alignment, so a straight line between its end shafts (OpenStreetMap
# ways 169957751 at Abbey Mills and 169957756 at Beckton)
feats.append(
    feat(
        LineString([ll(-0.0024072, 51.5284626), ll(0.0816377, 51.5196717)]),
        name="Lee Tunnel: Abbey Mills to Beckton",
        opened=2016,
        mode="sewer",
        note="6.9 km, about 75 m deep. Built 2010-16; carries the Tideway's flows on to Beckton sewage works",
        source="OpenStreetMap shaft locations",
        confidence="indicative: straight line between the end shafts",
    )
)
save(feats, "tunnels_utility")
for name, geom in gpd.read_file(LAYERS / "tunnels_utility.geojson").to_crs(BNG)[["name", "geometry"]].values:
    print(f"  {name}: {geom.length / 1000:.1f} km inside the circle")

  tunnels_utility.geojson: 3 features
  Thames Tideway Tunnel (main tunnel): 11.2 km inside the circle
  Thames Tideway Tunnel: Greenwich connection tunnel: 4.6 km inside the circle
  Lee Tunnel: Abbey Mills to Beckton: 4.4 km inside the circle


## TfL 2017 tunnel option

TfL's 2017 shortlist included an immersed tube tunnel (about GBP 440m whole-life NPV, against
GBP 225-300m for a bridge), but TfL published no route for it: Factsheet 2 says only that it "could be
relatively flexible in location with no need to construct entrances directly on the riverbank".
IanVisits' write-up of the consultation lands it in two docks, with sloping entrances built in
the docks and the docks reinstated above them: the Nelson Dock basin on the Rotherhithe side and the
West India Docks impound lock inlet off Westferry Road, as drawn on the article's map. The route is
traced from that map, not from a TfL drawing.


In [ ]:
IANVISITS = "IanVisits, 'Pedestrian and cyclist crossing planned for the Thames' (8 Nov 2017)"
TFL_TUNNEL_NOTE = "Immersed tube. TfL 2017: about GBP 440m whole-life NPV (2016 prices); 1.6m walking and up to 0.9m cycling trips a year by 2031"

tfl17_tunnel = [
    feat(
        LineString([NELSON_HEAD, NELSON_MOUTH]),
        name="TfL 2017 immersed tube: south ramp in Nelson Dock",
        element="ramp",
        note="'On the Rotherhithe side, the dock wall is at street level.' Dock reinstated above the ramp",
        source=IANVISITS,
        confidence="indicative",
    ),
    feat(
        LineString([NELSON_MOUTH, INLET_MOUTH]),
        name="TfL 2017 immersed tube: river crossing",
        element="alignment",
        note=TFL_TUNNEL_NOTE,
        source=f"{IANVISITS}; TfL consultation Factsheet 2: Crossing options (2017)",
        confidence="indicative",
    ),
    feat(
        LineString([INLET_MOUTH, INLET_HEAD]),
        name="TfL 2017 immersed tube: north ramp in the impound lock inlet",
        element="ramp",
        note="'On the Canary Wharf side ... the inlet dock could be a viable access point'",
        source=IANVISITS,
        confidence="indicative",
    ),
    feat(
        Point(NELSON_HEAD),
        name="TfL 2017 immersed tube: south entrance, Nelson Dock",
        element="portal",
        note="Rotherhithe Street, next to the Doubletree hotel",
        source=IANVISITS,
        confidence="site suggested, position indicative",
    ),
    feat(
        Point(INLET_HEAD),
        name="TfL 2017 immersed tube: north entrance, Westferry Road",
        element="portal",
        note="At the impound lock, just north of Cuba Street",
        source=IANVISITS,
        confidence="site suggested, position indicative",
    ),
]
save(tfl17_tunnel, "tunnel_tfl_2017")

  tunnel_tfl_2017.geojson: 5 features


## Proposed tunnel

The route in the blog, from the config.

In [ ]:
save(
    [
        feat(
            tunnel_line(),
            name="Proposed tunnel: Durand's Wharf to the impound lock",
            note="Walking and cycling tunnel: a 3.5 m cycle lane beside a 2.5 m footway, about 460 m portal to portal",
            source="This project (tunnel.config)",
            confidence="indicative",
        ),
        feat(
            Point(to_bng(SOUTH_PORTAL)),
            name="Proposed tunnel: south portal, Durand's Wharf park",
            note="Council-owned park on Rotherhithe Street; TfL's 2019 bridge design landed here too",
            source="This project (tunnel.config)",
            confidence="site proposed, position indicative",
        ),
        feat(
            Point(to_bng(NORTH_PORTAL)),
            name="Proposed tunnel: north portal, impound lock",
            note="Land by the West India Docks impound lock on Westferry Road",
            source="This project (tunnel.config)",
            confidence="site proposed, position indicative",
        ),
    ],
    "tunnel_proposed",
)

  tunnel_proposed.geojson: 3 features


### Clearance from the Jubilee line

The TfL 2017 immersed tube crosses the Jubilee line tunnels in plan, so it would have to pass over them.


In [ ]:
jubilee = gpd.read_file(LAYERS / "tunnels_existing.geojson").to_crs(BNG)
jubilee = jubilee[jubilee.name.str.startswith("Jubilee line")].geometry.iloc[0]
tube = gpd.read_file(LAYERS / "tunnel_tfl_2017.geojson").to_crs(BNG)
for _, r in tube[tube.element == "alignment"].iterrows():
    crosses = r.geometry.intersects(jubilee)
    print(
        f"{r['name']}: {r.geometry.length:.0f} m long, "
        + (
            "crosses the Jubilee line"
            if crosses
            else f"{r.geometry.distance(jubilee):.0f} m clear of the Jubilee line at closest"
        )
    )

TfL 2017 immersed tube: river crossing: 384 m long, crosses the Jubilee line


## Map

Builds `crossing_map/crossing_map.html` from every GeoJSON in `layers/`. Files not in `LAYER_STYLES`
still appear, with a default style. Each tunnel gets a label with its name, when it was built and
what it carries (`TUNNEL_LABELS`), pointing at where it crosses the river.


In [ ]:
OUT = MAP_DIR / "crossing_map.html"
CENTRE = (51.5035, -0.0300)

# file stem -> (layer name, colour, shown on load). Order is layer-control order.
LAYER_STYLES = {
    "ownership": (
        f"Land ownership (HMLR, readjustment area within {CATCHMENT_M} m)",
        None,
        True,
    ),
    "tunnel_proposed": ("Tunnel: proposed, Durand's Wharf to the impound lock", COLOURS["tunnel"], True),
    "tunnel_tfl_2017": (
        "Tunnel: TfL 2017 immersed tube option (route per IanVisits)",
        "#08519c",
        True,
    ),
    "tunnels_existing": ("Road, rail and foot tunnels", "#7b3294", True),
    "tunnels_historic": ("Abandoned Thames Archway tunnel (1805-09)", "#8c6d31", True),
    "tunnels_utility": ("Sewer tunnels", "#4d9221", True),
}
DEFAULT_COLOUR = "#3366cc"

# Rail lines in tunnels_existing get their own colours
LINE_COLOURS = {
    "Jubilee": "#7b868c",
    "London Overground": "#ee7c0e",
    "DLR": "#00a4a7",
    "Elizabeth line": "#6950a1",
}
MODE_COLOURS = {
    "road": "#8c510a",
    "foot": "#c51b7d",
    "sewer": "#4d9221",
}

POPUP_SKIP = {"colour", "group", "freehold"}


def line_style(colour: str, props: dict) -> dict:
    """Dashed = proposed, dotted = abandoned, solid = built; width follows width_m."""
    width = props.get("width_m")
    style = {
        "color": colour,
        "weight": 1.5 + 0.6 * width if width else 3,
        "opacity": 0.85,
    }
    if props.get("status") == "proposed":
        style["dashArray"] = "10 7"
    elif props.get("status") == "abandoned":
        style["dashArray"] = "2 6"
    return style


def popup_html(props: dict) -> str:
    rows = "".join(
        f"<tr><th style='text-align:left;padding-right:6px;vertical-align:top'>{k}</th><td>{v}</td></tr>"
        for k, v in props.items()
        if k not in POPUP_SKIP and v not in (None, "", "nan")
    )
    return f"<table style='font-size:12px'>{rows}</table>"


def add_vector_layer(m, gdf: gpd.GeoDataFrame, name: str, colour: str, show: bool) -> None:
    fg = folium.FeatureGroup(name=name, show=show)
    for _, row in gdf.iterrows():
        props = {k: v for k, v in row.drop("geometry").items() if not (isinstance(v, float) and math.isnan(v))}
        c = colour
        if "line" in props:
            c = LINE_COLOURS.get(props["line"], colour)
        elif props.get("mode") in MODE_COLOURS:
            c = MODE_COLOURS[props["mode"]]
        geom = row.geometry
        tooltip = props.get("name") or props.get("note")
        popup = folium.Popup(popup_html(props), max_width=380)
        if geom.geom_type == "Point":
            folium.CircleMarker(
                (geom.y, geom.x),
                radius=7,
                color=c,
                weight=2,
                fill=True,
                fill_color="white" if props.get("status") != "built" else c,
                fill_opacity=1,
                tooltip=tooltip,
                popup=popup,
            ).add_to(fg)
        else:
            style = line_style(c, props)
            if geom.geom_type in ("Polygon", "MultiPolygon"):
                style.update(fillColor=c, fillOpacity=0.15, weight=2)
            layer = folium.GeoJson(
                geom.__geo_interface__,
                style_function=lambda _, s=style: s,
                tooltip=tooltip,
            )
            popup.add_to(layer)
            layer.add_to(fg)
    fg.add_to(m)


def add_ownership(m, path, name: str, show: bool) -> None:
    gdf = gpd.read_file(path)
    fields = [
        "owner_group",
        "owner",
        "tenure",
        "title_no",
        "address",
        "owner_category",
        "area_m2",
    ]
    folium.GeoJson(
        json.loads(gdf.to_json()),
        name=name,
        show=show,
        style_function=lambda f: {
            "fillColor": f["properties"]["colour"],
            "fillOpacity": 0.55,
            "color": "white",
            "weight": 0.5,
        },
        highlight_function=lambda _: {"weight": 2, "color": "black"},
        tooltip=folium.GeoJsonTooltip(fields=["owner_group", "owner"], aliases=["", ""], labels=False),
        popup=folium.GeoJsonPopup(
            fields=fields,
            aliases=[
                "Group",
                "Owner",
                "Tenure",
                "Title",
                "Address",
                "Category",
                "Area (m²)",
            ],
        ),
    ).add_to(m)

    legend = [(label, COLOURS[k]) for k, label in GROUPS.items() if label in set(gdf.owner_group)]
    items = "".join(
        f"<div><span style='display:inline-block;width:12px;height:12px;background:{c};margin-right:6px'></span>{g}</div>"
        for g, c in legend
    )
    html = f"""
    <div style="position:fixed;bottom:24px;left:12px;z-index:9999;background:white;padding:8px 10px;
                border-radius:4px;box-shadow:0 1px 4px rgba(0,0,0,.3);font:12px sans-serif">
      <b>Registered owner</b>{items}
      <div style="margin-top:6px;color:#555">Lines: solid built, dashed proposed, dotted abandoned.<br>
      Thicker lines are wider bores. Click a feature<br>for its source and whether its path is approximate.</div>
    </div>"""
    m.get_root().html.add_child(folium.Element(html))


def add_context(m) -> None:
    """Catchment boundary and stations, from the existing ownership and accessibility data."""
    buf = gpd.GeoSeries([catchment()], crs=BNG).to_crs(WGS84)
    fg = folium.FeatureGroup(name=f"{CATCHMENT_M} m readjustment area", show=True)
    folium.GeoJson(
        buf.boundary.iloc[0].__geo_interface__,
        style_function=lambda _: {"color": "#555", "weight": 1.5, "dashArray": "4 4"},
    ).add_to(fg)
    fg.add_to(m)

    fg = folium.FeatureGroup(name=f"Tunnel circle ({TUNNEL_RADIUS_M / 1000:.1f} km)", show=True)
    folium.GeoJson(
        gpd.GeoSeries([TUNNEL_AREA.boundary], crs=BNG).to_crs(WGS84).iloc[0].__geo_interface__,
        style_function=lambda _: {"color": "#888", "weight": 1, "dashArray": "1 5"},
    ).add_to(fg)
    fg.add_to(m)

    st = stations().to_crs(WGS84)
    fg = folium.FeatureGroup(name="Stations and piers", show=False)
    for _, r in st.iterrows():
        p = r.geometry.representative_point()
        folium.CircleMarker(
            (p.y, p.x),
            radius=4,
            color="#222",
            weight=1,
            fill=True,
            fill_color="#fff",
            fill_opacity=1,
            tooltip=r.get("name"),
        ).add_to(fg)
    fg.add_to(m)

In [ ]:
# (name prefix of its features, title, when built, what it carries, colour, label offset in px)
# Offsets are tuned at zoom 15; below LABEL_MIN_ZOOM the labels hide rather than overlap.
LABEL_MIN_ZOOM = 15
TUNNEL_LABELS = [
    (
        "Proposed tunnel: Durand",
        None,
        "Proposed tunnel",
        "Durand's Wharf to the impound lock",
        "Pedestrians and cyclists",
        COLOURS["tunnel"],
        (-40, -60),
    ),
    # (name prefix, near (lon, lat) or None, title, built, carries, colour, offset)
    (
        "Thames Tunnel",
        None,
        "Thames Tunnel",
        "Built 1825–43",
        "London Overground trains; a foot tunnel until 1865",
        LINE_COLOURS["London Overground"],
        (-30, 60),
    ),
    (
        "Rotherhithe Tunnel",
        None,
        "Rotherhithe Tunnel",
        "Built 1904–08",
        "Cars and vans, cycles, pedestrians",
        MODE_COLOURS["road"],
        (-30, -70),
    ),
    (
        "Thames Archway",
        None,
        "Thames Archway (abandoned)",
        "Dug 1805–08",
        "Pilot driftway; never carried traffic",
        LAYER_STYLES["tunnels_historic"][1],
        (40, -70),
    ),
    (
        "Jubilee line",
        (-0.02964, 51.50396),
        "Jubilee line",
        "Built 1993–99",
        "Tube trains",
        LINE_COLOURS["Jubilee"],
        (60, -55),
    ),
    (
        "TfL 2017 immersed tube: river",
        None,
        "TfL 2017 immersed tube (option)",
        "Shortlisted 2017, not built",
        "Pedestrians and cyclists",
        LAYER_STYLES["tunnel_tfl_2017"][1],
        (30, 60),
    ),
    (
        "Jubilee line",
        (-0.00507, 51.50043),
        "Jubilee line",
        "Built 1993–99",
        "Tube trains",
        LINE_COLOURS["Jubilee"],
        (-50, 60),
    ),
    (
        "Elizabeth line",
        (-0.0098, 51.50614),
        "Elizabeth line",
        "Built 2009–22",
        "Elizabeth line trains",
        LINE_COLOURS["Elizabeth line"],
        (-40, -60),
    ),
    (
        "Blackwall Tunnel",
        None,
        "Blackwall Tunnel",
        "Built 1892–97 and 1960–67",
        "Motor vehicles only",
        MODE_COLOURS["road"],
        (60, -60),
    ),
    (
        "Jubilee line",
        (0.00994, 51.50407),
        "Jubilee line",
        "Built 1993–99",
        "Tube trains",
        LINE_COLOURS["Jubilee"],
        (40, -50),
    ),
    (
        "Silvertown Tunnel",
        None,
        "Silvertown Tunnel",
        "Built 2020–25",
        "Motor vehicles, with a bus and lorry lane",
        MODE_COLOURS["road"],
        (50, 40),
    ),
    (
        "DLR",
        (-0.00844, 51.48513),
        "DLR Lewisham extension",
        "Built 1996–99",
        "DLR trains",
        LINE_COLOURS["DLR"],
        (50, -40),
    ),
    (
        "Greenwich foot tunnel",
        None,
        "Greenwich foot tunnel",
        "Built 1899–1902",
        "Pedestrians; cyclists must walk",
        MODE_COLOURS["foot"],
        (-50, 40),
    ),
    (
        "Thames Tideway Tunnel (main",
        None,
        "Thames Tideway Tunnel",
        "Built 2016–25",
        "Storm sewage, to Beckton for treatment",
        MODE_COLOURS["sewer"],
        (-70, -75),
    ),
    (
        "Thames Tideway Tunnel: Greenwich",
        None,
        "Tideway: Greenwich connection",
        "Built 2016–25",
        "Storm sewage from Deptford and Greenwich",
        MODE_COLOURS["sewer"],
        (-50, 50),
    ),
    (
        "Lee Tunnel",
        None,
        "Lee Tunnel",
        "Built 2010–16",
        "Storm sewage, Abbey Mills to Beckton",
        MODE_COLOURS["sewer"],
        (40, -50),
    ),
    (
        "Woolwich foot tunnel",
        None,
        "Woolwich foot tunnel",
        "Built 1910–12",
        "Pedestrians; cyclists must walk",
        MODE_COLOURS["foot"],
        (-50, -40),
    ),
]

LABEL_CSS = """<style>
.tunnel-label { pointer-events: none; }
.hide-tunnel-labels .tunnel-label { display: none; }
.tunnel-label .card { position: absolute; white-space: nowrap; background: rgba(255,255,255,.93);
  padding: 3px 7px 4px; border-radius: 3px; box-shadow: 0 1px 3px rgba(0,0,0,.25);
  font: 11px/1.35 -apple-system, "Segoe UI", sans-serif; color: #555; }
.tunnel-label .card b { display: block; font-size: 12px; color: #222; }
</style>"""


def label_anchor(gdf: gpd.GeoDataFrame, river, near=None) -> Point:
    """The point on the features nearest `near` if given; else where they cross the river centre line,
    else the middle of the line.
    Twin bores get the point on a bore nearest the middle of their crossings; tunnels that
    cross many times along the river (Tideway) get the crossing nearest the map centre."""
    lines = gdf.geometry[gdf.geom_type != "Point"].union_all()
    if near:
        return nearest_points(lines, Point(ll(*near)))[0]
    crossing = lines.intersection(river)
    if crossing.is_empty:
        return shapely.line_merge(lines).interpolate(0.5, normalized=True)
    if shapely.minimum_bounding_radius(crossing) > 300:
        return nearest_points(crossing, Point(ll(CENTRE[1], CENTRE[0])))[0]
    return nearest_points(lines, crossing.centroid)[0]


def add_tunnel_labels(m) -> None:
    tunnels = pd.concat(
        gpd.read_file(LAYERS / f"{stem}.geojson").to_crs(BNG)
        for stem in (
            "tunnels_existing",
            "tunnels_historic",
            "tunnel_tfl_2017",
            "tunnel_proposed",
            "tunnels_utility",
        )
    )
    river = linemerge(thames().union_all())
    fg = folium.FeatureGroup(name="Tunnel labels", show=True)
    for prefix, near, title, built, traffic, colour, (dx, dy) in TUNNEL_LABELS:
        match = tunnels[tunnels.name.str.startswith(prefix)]
        pt = gpd.GeoSeries([label_anchor(match, river, near)], crs=BNG).to_crs(WGS84)[0]
        shift = "-100%" if dx < 0 else "0"
        html = f"""
        <svg width="1" height="1" style="overflow:visible;position:absolute">
          <line x1="0" y1="0" x2="{dx}" y2="{dy}" stroke="{colour}" stroke-width="1.2"/>
          <circle r="3" fill="white" stroke="{colour}" stroke-width="1.5"/>
        </svg>
        <div class="card" style="left:{dx}px;top:{dy}px;transform:translate({shift},-50%);border-left:3px solid {colour}">
          <b>{title}</b>{built}<br>{traffic}
        </div>"""
        folium.Marker(
            (pt.y, pt.x),
            icon=folium.DivIcon(html=html, icon_size=(0, 0), class_name="tunnel-label"),
            interactive=False,
        ).add_to(fg)
    fg.add_to(m)
    m.get_root().header.add_child(folium.Element(LABEL_CSS))
    # The map variable is only defined once folium's own script has run.
    toggle = f"""document.addEventListener("DOMContentLoaded", function () {{
        {m.get_name()}.on("zoomend", function () {{
            this.getContainer().classList.toggle("hide-tunnel-labels", this.getZoom() < {LABEL_MIN_ZOOM});
        }});
    }});"""
    m.get_root().script.add_child(folium.Element(toggle))

In [ ]:
m = folium.Map(location=CENTRE, zoom_start=15, tiles=None, control_scale=True)
# Esri tiles need no API key (CARTO's now do). cross_origin lets tiles load in Cursor/VS Code
# notebook output, which blocks cross-origin images that aren't requested with CORS.
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    attr="Esri, HERE, Garmin, OpenStreetMap contributors",
    name="Light basemap",
    max_zoom=20,
    max_native_zoom=16,
    cross_origin=True,
).add_to(m)
folium.TileLayer("OpenStreetMap", name="OpenStreetMap", show=False, cross_origin=True).add_to(m)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery",
    name="Satellite",
    max_zoom=20,
    max_native_zoom=19,
    show=False,
    cross_origin=True,
).add_to(m)

files = {p.stem: p for p in sorted(LAYERS.glob("*.geojson"))}
ordered = [s for s in LAYER_STYLES if s in files] + [s for s in files if s not in LAYER_STYLES]
for stem in ordered:
    name, colour, show = LAYER_STYLES.get(stem, (stem.replace("_", " ").capitalize(), DEFAULT_COLOUR, True))
    if stem == "ownership":
        add_ownership(m, files[stem], name, show)
    else:
        add_vector_layer(m, gpd.read_file(files[stem]), name, colour or DEFAULT_COLOUR, show)
    print(f"  layer: {name}")

add_context(m)
add_tunnel_labels(m)

Draw(
    export=True,
    filename="annotations.geojson",
    position="topleft",
    draw_options={"circlemarker": False},
).add_to(m)
MeasureControl(position="topleft", primary_length_unit="meters", primary_area_unit="sqmeters").add_to(m)
MousePosition(position="bottomright", separator=", ", num_digits=6, prefix="lat, lon:").add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

title = """<div style="position:fixed;top:10px;left:60px;z-index:9999;background:white;padding:6px 10px;
           border-radius:4px;box-shadow:0 1px 4px rgba(0,0,0,.3);font:bold 14px sans-serif">
           Rotherhithe to Canary Wharf crossing</div>"""
root = cast(Figure, m.get_root())
root.html.add_child(folium.Element(title))
root.header.add_child(folium.Element("<title>Rotherhithe to Canary Wharf crossing</title>"))

m.save(OUT)
print(f"Wrote {OUT.relative_to(ROOT)}")

  layer: Land ownership (HMLR, readjustment area within 750 m)
  layer: Tunnel: proposed, Durand's Wharf to the impound lock
  layer: Tunnel: TfL 2017 immersed tube option (route per IanVisits)
  layer: Road, rail and foot tunnels
  layer: Abandoned Thames Archway tunnel (1805-09)
  layer: Sewer tunnels


Wrote crossing_map/crossing_map.html


In [ ]:
root.height = "700"
m